<a href="https://colab.research.google.com/github/Arnavsao/Fine-Tuning-Llama-Model-2-on-Intruction-dataset/blob/main/Fine_Tuning_Llama_2_Model_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install All the Required Packages

In [1]:
!pip install -q accelerate peft bitsandbytes transformers trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 35.8 MB/s eta 0:00:00


In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from google.colab import userdata
from huggingface_hub import login


token = userdata.get('HUGGINGFACEHUB_API_TOKEN')
login(token)

# **In case of Llama 2 the following prompt template is used for the chat models**

System prompt (optional) to guide the model

User prompt (req) to give the instructions

Model Answer (req)

```
<s>[INST] <<SYS>>
{System Prompt}
<</SYS>>

{User Prompt}
[/INST]

{Model Answer}
</s>
```

# **We will reformat our instruction dataset to follow llama 2 template**

Orignal Dataset: https://huggingface.co/datasets/timdettmers/openassistant-guanaco

Reformat Dataset following the llama 2 template with 1k sample:https://huggingface.co/datasets/mlabonne/guanaco-llama2-1k

Complete Reformal dataset following the llama 2 template: https://huggingface.co/datasets/mlabonne/guanaco-llama2

To know how this dataset was created, you can check this notebook.
https://colab.research.google.com/drive/11oUe11VosVhHjXIWzRoXSuZJ-3297xZY

Note: You Don't need to follow a specific prompt template if you're using the instead of the base llama 2 model chat version. here we are not.

# **How to fine tune Llama 2**

* Free Google Colab offer 15GB graphics Card(Limited Resource---> BArely enough to store Llama 2 -7b's weights)

* We also need to consider the overhead due to optimizer state, gradients and over forward activations.

* Full fine tuning is not possible here we need parameter efficient fine tuning (PEFT) technique like LoRA or QLoRA.

* To drastically reduce the VRAM usage, we must fine tune the model in 4 bit precision, which is why we'll use QLoRA here.



#**Step 3**
1. Load a llama -2-7b-chat-hf model (chat model)

2. Train it on the mlabonne/guanaco-llama2-1k (1000 samples), which will produce out fine-tuned model Llama-2-7b-chat-finetune.

OLoRA will use a rank of 64 with a scaling parameter of 16. We'll load the Llama 2 model directly in 4-bit precision using the NF4 type and train for one epoch

In [3]:
# The model that you want to train from the Hugging Face ub
model_name = "meta-llama/Llama-2-7b-chat-hf"

# The instruction dataset to use
dataset_name = "mlabonne/guanaco-llama2-1k"

# Fine-tuned model name
new_model = "llama-2-7b-chat-finetune"


# QLoRA Parameters

# LoRA attention dimension
lora_r = 64

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1



# bitsandbyte parameters

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4bit base model
bnb_4bit_compute_dtype = "float16"

# Quanlization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4 bit base models (double quantixation)
use_nested_quant = False



# Training Parameters

#Ouput directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 1

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 1
gradient_accumulation_steps = 4

# Number of update steps to accumulate the gradient for
gradient_accumulation_steps = 1

#Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

#Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# learningrate schedule
Ir_scheduler_type = "cosine"

# Number f training steps (override num_train_epochs)
max_steps = -1

#Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequence into batches with same length
# Save memory and speeds up training considerably
group_by_length = True

#save checkpoints every X updates steps
save_steps =0

#log every X updates steps
logging_steps =25



# SFT parameters

# Maximum sequence length to use
max_seq_length = None

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model to the GPU 0
device_map = {"":0}


# **Step 4: Load everything and start the fine tuning process**

1. First of all, we want o load the dataset we defined. Here, out dataset is already preprocessed but, usuall, this is where you would reformat the prompt, filter out bad text, combine multiple datasets, etc.

2. Then, we're configuring bitsandbytes for 4-bit quantization.

3. Next, we're oading the Llama 2 a model in 4-bit precision on a GPU with the corresponding tokenizer.

4. Finally we're loading cofigurations for QLoRA, regular training parameters, and passing everything to the SFTTrainer. The training can finally start.

In [ ]:
# Loading dataset (you can process it here)
dataset = load_dataset(dataset_name, split="train")

# Load tokenizer
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# check GPU compatibility with bfloat16
if torch.cuda.is_available() and compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # fix weird overflow issue with fp16 training

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# set training parameter
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=Ir_scheduler_type,
    report_to="tensorboard"
)

# Set supervised fine tuning parameter
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_arguments,
    max_seq_length=max_seq_length,
    packing=packing,
)

# Train Model
trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]